<a href="https://colab.research.google.com/github/whiplaaash/TF-NLP/blob/main/colab_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to NLP fundamentals in Tensor Flow

NLP has the goal of deriving information out of natural language (could be sequences, text or speach).

Another common term for NLP problems is sequence to seqence (seq2seq).

# Check for GPU (to change from CPU to GPU use "Srodowisko wykonawcze" and then "Zmien typ srodowiska wykonawczego").


In [2]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-2f6e3bfa-4409-abda-b254-2610dfc8fdc9)


# Get helper functions

In [3]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py

--2026-11-11 15:50:09--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-11-11 15:50:09 (95.5 MB/s) - ‘helper_functions.py’ saved [10246/10246]



# Import a series of helper functions for the notebook.

In [4]:
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

# Get a dataset

We are going to be using a dataset taken from Kaggle. It is an introduction to NLP dataset (text samples of Tweets labelled as a disaster or not a disaster).

Se the original source:
https://www.kaggle.com/competitions/nlp-getting-started

In [5]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip


--2026-11-11 15:50:14--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 192.179.26.207, 173.194.202.207, 173.194.203.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|192.179.26.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K  --.-KB/s    in 0.005s  

2026-11-11 15:50:14 (127 MB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



In [6]:
# Unzip data
unzip_data("nlp_getting_started.zip")

# Visualizing a text dataset

In [7]:
# We use pandas package to read the data.

import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df.head()


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [8]:
train_df["text"][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [9]:
# Shuffle training dataframe

train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [10]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [11]:
# How many examples of class are there?
train_df_shuffled.target.value_counts()

# Data are quite well balanced (60:40)

,count
target,
0,4342
1,3271


In [12]:
# How many total samples?
len(train_df), len(test_df)

(7613, 3263)

# Let's visualize some data

In [13]:
# Let's generate 10 random samples
import random
random_index = random.randint(0, len(train_df)-5)
for row in train_df_shuffled[["text", "target"]][random_index:random_index+5].itertuples():
# we generate random samples in the range [random_index:random_index+5]
# itertuples() - returns values in the row as a tuple
  _, text, target = row
  print(f"Target: {target}", "(real disaster)" if target==1 else "(not real disaster)")
  print(f"Text:\n{text}\n")
  print("-----\n")

Target: 1 (real disaster)
Text:
@SwellyJetEvo Disneyland! Tacos there are bomb!

-----

Target: 0 (not real disaster)
Text:
My blood pressure is through the roof I don't need all this extra shit!!!!

-----

Target: 0 (not real disaster)
Text:
Reddit Will Now Quarantine Offensive Content: Reddit co-founder and CEO Steve Huffman has unveiled more specif... http://t.co/T7gE0j3CAy

-----

Target: 0 (not real disaster)
Text:
Ronda Rousey would be 'close' to making Floyd Mayweather's money in 50 fights - Bloody Elbow http://t.co/IjzcYtbFfo #boxing

-----

Target: 0 (not real disaster)
Text:
@Camilla_33 @CrayKain Hate to shatter your delusions but a hatchet is a deadly weapon justifying lethal force. #gunsense

-----



In [14]:
# Split data into training and validation sets
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1, # use 10% of data as validation set
                                                                            random_state=42 # for reproduction of the data
                                                                            )

In [15]:
# Check lenghts
len(train_sentences), len(val_sentences), len(train_labels), len(val_labels)

(6851, 762, 6851, 762)

In [16]:
# check the first 10 samples
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

# Converting text to numbers (tokenization and embbeding)

Before building a model based on words, the first thing to do is to convert words to numbers.

**Tokenization**: converting tokens (words, subwords, characters) into numbers, usually with one-hot encoding.

**Embedding**: takes into account the context of w word, probability of beng close to other words. We may use pre-learnt embedding. For each token you create a matrix of feature vectors.

# Tokenization (text vectorization)

In [17]:
train_sentences[:5]

array(['@mogacola @zamtriossu i screamed after hitting tweet',
       'Imagine getting flattened by Kurt Zouma',
       '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
       "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
       'Somehow find you and I collide http://t.co/Ee8RpOahPk'],
      dtype=object)

In [18]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Use the default parameters
text_vectorizer = TextVectorization(max_tokens=None, # how many tokens in the whole set of vocabulary
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None, # groups of n number of words
                                    output_mode="int", # how to represent words as numbers
                                    output_sequence_length=None, # how long do you want your sequences to be
                                    pad_to_max_tokens=False)

In [19]:
# Find the avergae number of tokens in the training tweets
round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [20]:
# Setup text vectorization variables
max_vocab_length = 10000 # max number of words to have in our vocabulary
max_length = 15 # max length our sentences will be

# Use fixed, and no default, parameters
text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [21]:
# Fit the text vectorizer to the training data
text_vectorizer.adapt(train_sentences)

In [22]:
# Create a sample sentence and tokenize it
sample_sentence = "There is a flood on my street!"
text_vectorizer([sample_sentence])


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[ 74,   9,   3, 232,  11,  13, 698,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [23]:
# choose a random sentence from the training dataset and tokenize it
random_sentence = random.choice(train_sentences)
print(f"Original sentence:\n{random_sentence}\
      \n\nVectorized version:")
text_vectorizer([random_sentence])

Original sentence:
Doone Silver Architects has won permission to demolish Birmingham's Natwest Tower and replace it with what will be cityÛªs tallest building.      

Vectorized version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[   1, 1112,    1,   41, 1449, 9851,    5,  516,    1,    1, 7439,
           7, 1777,   15,   14]])>

In [24]:
# Get the unique words in vocabulary
words_in_vocab = text_vectorizer.get_vocabulary() # get all the unique words in the vocabulary
top_5_words = words_in_vocab[:5] # get the most common words
bottom_5_words = words_in_vocab[-5:] # get the least common words in vocab
print(f"Number of words in vocabulary: {len(words_in_vocab)}")
print(f"5 most common words: {top_5_words}")
print(f"5 least common words: {bottom_5_words}")


Number of words in vocabulary: 10000
5 most common words: ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
5 least common words: [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


# Embedding

To make an embedding we are going to use Tensor Flow embedding layer.

The parameters we care about most:
- `input_dim` - how many words are in vocabulary
- `output_dim` - the size of the output embedding vector, for example: a numbe 100 means that each token will be represented by a vector 100 long
- `input_length` - length of the sequences being passed to the embedding layer (this parameter no longer exist!)

In [25]:
from tensorflow.keras import layers
embedding = layers.Embedding(input_dim=max_vocab_length, #set the input shape
                             output_dim=128 # something divisible by 8, output shape
                             )
embedding

<Embedding name=embedding, built=False>

In [26]:
# Get a random sentence from the trainig set
random_sentence = random.choice(train_sentences)
print(f"Original sentence:\n{random_sentence}\
\n\nEmbedded version:")

# Embed a random sentence (turn it into dense vectors of fixed size)
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

Original sentence:
@Bang_Me_Up_Guk he was ;-; like he was singing so loud ;-;

Embedded version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[-0.01271448,  0.01909499,  0.00213435, ..., -0.04366991,
         -0.01637036, -0.01767063],
        [ 0.02765814,  0.04014069, -0.01266869, ...,  0.03939787,
          0.02398671, -0.03355797],
        [ 0.04855843,  0.04546392, -0.02114989, ..., -0.00763601,
          0.01242441,  0.04131847],
        ...,
        [-0.02392119,  0.00930528,  0.02964618, ...,  0.01341275,
         -0.0075077 , -0.0352766 ],
        [-0.02392119,  0.00930528,  0.02964618, ...,  0.01341275,
         -0.0075077 , -0.0352766 ],
        [-0.02392119,  0.00930528,  0.02964618, ...,  0.01341275,
         -0.0075077 , -0.0352766 ]]], shape=(1, 15, 128), dtype=float32)>

In [27]:
# Check out a single token's embedding
sample_embed[0][0], sample_embed[0][0].shape, random_sentence[0]

# This array represents the first word in the random sentence
# The array shows random numbers which will be updated when the model learns
# The length of each suach array is 128 and is fixed

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([-0.01271448,  0.01909499,  0.00213435,  0.01276573,  0.00613053,
         0.03140889, -0.01415221,  0.04171645,  0.00560393,  0.03891209,
        -0.01610129,  0.03839395,  0.01492944, -0.02257401,  0.00845846,
         0.01359538,  0.01265533,  0.02880352, -0.03791086,  0.04690727,
        -0.02803624, -0.02575606,  0.00183968,  0.04882928, -0.03565922,
         0.03196405,  0.04404993,  0.04681975,  0.02310859, -0.01048472,
        -0.00449985,  0.04803706,  0.00819387, -0.02417753, -0.03886201,
        -0.04808165,  0.01384285,  0.02054035, -0.02194796,  0.04511775,
        -0.01693946,  0.04981362, -0.04619103, -0.01074194,  0.01493373,
        -0.00012954, -0.0149614 ,  0.03477787, -0.04805284,  0.00942919,
        -0.03962041, -0.02671068,  0.01725263,  0.0479813 ,  0.04591776,
        -0.02609774, -0.04767166,  0.00504037,  0.04565363,  0.04956124,
         0.03472586, -0.02726666,  0.01660682,  0.01363487,  0.00839071,
  

# Modelling a text dataset (running a series of experiments)

After turning a text into numbers we will be running 8 experiments. We will start with a baseline and will be improving it.

* Model 0: Naive Bayes
* Model 1: Feed-forward neural network (dense model)
* Model 2: LSTM model (RNN)
* Model 3: GRU model (RNN)
* Model 4: Bidirectional LSTM model (RNN)
* Model 5: 1D Convolutional Neural Network model (CNN)
* Model 6: Tensor Flow Hub Pretrined Feature Extractor (using transfer learning for NLP)
* Model 7: Same as model 6 but with 10% of training data

To do all this we will go through the standard procedure of machine  learning:
- create a model
- build a model
- fit a model
- evaluate a model


# Model 0: Naive Bayes

This is a non-deep-learning model. It is part of Machine Learning and it comes from sklearn package.

⭕ It is important to start with a baseline model and then improve it. We will start with a Multinomial Naive Bayes with TF-IDF formula.

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [29]:
# Create tokanization and modelling pipeline
model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()), # convert words to numbers using tfidf
    ("clf", MultinomialNB()) # model the text, "clf" means a classifier
])

# Fir the pipeline to the training data
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [30]:
# Evaluate our baseline model
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Our baseline model achieves an accuracy of: {baseline_score*100:.2f}%")

Our baseline model achieves an accuracy of: 79.27%


This accuracy is more than a guess, which would be around 55%.

In [31]:
# Make predictions
baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1])

In [32]:
train_labels

array([0, 0, 1, ..., 1, 1, 0], shape=(6851,))

# Create a function which will be tracking the most important metrics across all models

Metrics:
- accuracy (not the best metric for imbalanced classes)
- precision (higher precison leads to less false positives)
- recall (higher recall leads to less false negatives)
- F1 score (combination of precision and recall)

In [33]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# similarly one could use tensorflow metrics instead of sklearn metrics

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall, and f1-score of a binary classification model.
  """
  # Calculate model accuracy:
  model_accuracy = accuracy_score(y_true, y_pred)*100
  # Calculate model precision, recall, f1-score using weighted average (weighted by support, that is, the number of cases in each class)
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  # the empty underscore "_" above stands for the support, but we are not interested in it
  model_results = {
      "accuracy": model_accuracy,
      "precision": model_precision,
      "recall": model_recall,
      "f1-score": model_f1}
  return model_results


In [34]:
# Get baseline results
baseline_results = calculate_results(y_true=val_labels, y_pred=baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1-score': 0.7862189758049549}

# Model 1: Simple dense model

⭐ Reminder: **callbacks** run some code at specific points of the code to do some extra operations on top of the running model:
- stop training when performance is not improving, [EarlyStopping],
- saving model during training [ModleCheckpoint],
- log metrics for visualization [TensorBoard],
- lower the training rate when progress stalls [ReduceLROnPlateau],
- change learning rate over time [LearningRateScheduler]

In [35]:
# Create a tensorboard callback (need to create a new one for each model)
from helper_functions import create_tensorboard_callback

# Create a directory to save TensorBoard logs
SAVE_DIR = "model_logs"

# Build model with the Functional API
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype=tf.string) # inputs are 1D strings
x = text_vectorizer(inputs) # turn the input text into numbers
x = embedding(x) # create an embedding of the numberized inputs
x = layers.GlobalAveragePooling1D()(x) # this converts a sequence into a single vector so that ndim of predictions and labels is the same
# pooling makes the model predicts one value per sentence, and per token
outputs = layers.Dense(1, activation="sigmoid")(x) # create an output layer, we want binary outputs so we use sigmoid activation function

model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")


In [36]:
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,280,129 (4.88 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

In [37]:
# Compile the model
model_1.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [38]:
# Fit the model
model_1_history = model_1.fit(x=train_sentences,
                              y=train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                         experiment_name="model_1_dense")])

Saving TensorBoard log files to: model_logs/model_1_dense/20261111-155020
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.6908 - loss: 0.6101 - val_accuracy: 0.7703 - val_loss: 0.5361
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8124 - loss: 0.4453 - val_accuracy: 0.7822 - val_loss: 0.4713
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8608 - loss: 0.3514 - val_accuracy: 0.7861 - val_loss: 0.4769
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8891 - loss: 0.2892 - val_accuracy: 0.7953 - val_loss: 0.4648
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9108 - loss: 0.2420 - val_accuracy: 0.7808 - val_loss: 0.4763


In [39]:
# check the results
model_1.evaluate(val_sentences, val_labels)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7808 - loss: 0.4763


[0.4763392508029938, 0.7808399200439453]

In [40]:
# Make some predictions and evaluate those
model_1_pred_probs = model_1.predict(val_sentences)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [41]:
model_1_pred_probs.shape

(762, 1)

In [42]:
# Look at the single prediction
model_1_pred_probs[0]

array([0.4342469], dtype=float32)

In [43]:
# Look at the first 10 predictions
model_1_pred_probs[:10]

array([[0.4342469 ],
       [0.7891661 ],
       [0.99746335],
       [0.17529206],
       [0.12696251],
       [0.9470206 ],
       [0.92000437],
       [0.9937133 ],
       [0.97044367],
       [0.32784003]], dtype=float32)

In [44]:
# To compare our preds to the val labels wee need to first convert them to the val format, which is to have 0 and 1 values
# First we squeeze the preds to cancel one extra dimension
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:20]

<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 1.], dtype=float32)>

In [45]:
# Calculate our model_1 results using the function that we defined before
model_1_results = calculate_results(y_true=val_labels, y_pred=model_1_preds)
model_1_results

{'accuracy': 78.08398950131233,
 'precision': 0.783783808499639,
 'recall': 0.7808398950131233,
 'f1-score': 0.7783998521836788}

In [46]:
# Comapare with the baseline results
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1-score': 0.7862189758049549}

In [47]:
import numpy as np
np.array(list(model_1_results.values())) > np.array(list(baseline_results.values()))

array([False, False, False, False])

It appears that the baseline results are better than the results coming from the dense neural network trainig.

# Visualizing learned embeddings

In [48]:
# Get the vocabulary from the text vectorization layer
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:10]

(10000,
 ['',
  '[UNK]',
  np.str_('the'),
  np.str_('a'),
  np.str_('in'),
  np.str_('to'),
  np.str_('of'),
  np.str_('and'),
  np.str_('i'),
  np.str_('is')])

In [49]:
max_vocab_length

10000

In [50]:
# Model_1 summary
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,840,389 (14.65 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,560,260 (9.77 MB)

In [57]:
# Get the weigth matrix of embedding layer
# (these are numerical representations of each token in our training data)
embed_weights = model_1.get_layer("embedding").get_weights()[0]
embed_weights
print(embed_weights.shape) # same sa vocab size and embedding dim (output dim of our embedding layer)

(10000, 128)


Now we have got the embedding matrix our model has learned to represent our tokens. Let's see how we can visulaize it.

To do so, TensorFlow has a handy tool caller projector: https://projector.tensorflow.org

And TensorFlow also has an incredible guide on word embeddings themselves: https://www.tensorflow.org/text/guide/word_embeddings?hl=pl



In [58]:
# Create embedding files (we got this from TensorFlow word embedding documentation)
import io

out_v = io.open('vectors.tsv', 'w', encoding='utf-8')
out_m = io.open('metadata.tsv', 'w', encoding='utf-8')

for index, word in enumerate(words_in_vocab):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = embed_weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
out_v.close()
out_m.close()

In [59]:
# Download fiels from Colab to upload to projector
try:
  from google.colab import files
  files.download('vectors.tsv')
  files.download('metadata.tsv')
except Exception:
  pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

At this point you can upload the files to:
https://projector.tensorflow.org
to see the distribution of the weights (of words in vocab) in 3D.

You can see more on embeddings here:
- https://jalammar.github.io/illustrated-word2vec/
